# 08 Contract Value and GM Decision Support

## Purpose

This notebook connects the finalized receiving yard predictions to contract cost information so the model can support roster and contract decisions.

The performance forecast remains separate from compensation data. Salary and contract variables are not used to predict receiving yards. They are added only after the performance prediction is produced.

The analysis will compare predicted receiving production with contract cost to identify players whose expected production may be high or low relative to their financial commitment. The output is intended to support GM review and prioritization, not make an automatic personnel decision.

In [1]:
from pathlib import Path

import numpy as np
import polars as pl

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

# Project paths
processed_dir = Path("../data/processed")

# Load finalized modeling data
wr_model = pl.read_csv(
    processed_dir / "wr_model.csv"
)

print("Modeling data shape:", wr_model.shape)
print(
    "Season range:",
    wr_model["season"].min(),
    "to",
    wr_model["season"].max()
)
print(
    "Unique players:",
    wr_model["player_id"].n_unique()
)

assert wr_model.shape == (2129, 139), (
    f"Expected (2129, 139), found {wr_model.shape}"
)

assert wr_model["player_id"].n_unique() == 620, (
    f"Expected 620 players, found "
    f"{wr_model['player_id'].n_unique()}"
)

print("Finalized modeling dataset verified.")

Modeling data shape: (2129, 139)
Season range: 2012 to 2024
Unique players: 620
Finalized modeling dataset verified.


In [2]:
import nflreadpy as nfl

# Load nflverse contract data
contracts = nfl.load_contracts()

# Keep wide receivers with valid GSIS IDs
wr_contracts_valid = (
    contracts
    .filter(
        (pl.col("position") == "WR")
        & pl.col("gsis_id").is_not_null()
    )
)

# Keep one nested contract history per player
wr_player_histories = (
    wr_contracts_valid
    .select([
        "gsis_id",
        "player",
        "position",
        "contract_history"
    ])
    .unique(
        subset=["gsis_id"],
        maintain_order=True
    )
)

# Expand contract history into unique contract events
wr_contract_history = (
    wr_player_histories
    .explode("contract_history")
    .unnest("contract_history")
    .unique(
        subset=[
            "gsis_id",
            "team",
            "contract_type",
            "status",
            "year_signed",
            "yrs",
            "total",
            "apy",
            "guarantees",
            "amount_earned",
            "percent_earned",
            "effective_apy"
        ],
        maintain_order=True
    )
)

print("Unique WR players with contract history:", wr_player_histories.height)
print("Unique WR contract events:", wr_contract_history.height)

Unique WR players with contract history: 1626
Unique WR contract events: 7122


C:\Users\billf\AppData\Local\Temp\ipykernel_15100\361967553.py:33: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("contract_history")


In [5]:
# Define finalized performance model predictors

target_column = "next_season_receiving_yards"
group_column = "player_id"

features_to_drop = [
    "passing_cpoe",
    "passing_epa",
    "pacr",
]

predictor_columns = [
    column
    for column in wr_model.columns
    if column not in [
        target_column,
        group_column,
        *features_to_drop,
    ]
]

print("Predictor columns:", len(predictor_columns))
print("Target column:", target_column)

assert len(predictor_columns) == 134, (
    f"Expected 134 predictors, found {len(predictor_columns)}"
)

for feature in features_to_drop:
    assert feature not in predictor_columns

print("Predictor definition verified.")

Predictor columns: 134
Target column: next_season_receiving_yards
Predictor definition verified.


### Forward Prediction Design

To avoid evaluating contracts with in sample predictions, each season is predicted using a Random Forest trained only on earlier seasons.

The prediction is for the following season. Contract information is matched afterward using the outcome season and is not included in the performance model.

In [6]:
# Generate forward out of sample predictions by predictor season

forward_predictions = []

predictor_seasons = sorted(
    wr_model["season"].unique().to_list()
)

for predictor_season in predictor_seasons:

    train_data = wr_model.filter(
        pl.col("season") < predictor_season
    )

    test_data = wr_model.filter(
        pl.col("season") == predictor_season
    )

    # Skip the earliest season if no prior training data exists
    if train_data.height == 0:
        continue

    X_train_forward = train_data.select(
        predictor_columns
    )

    X_test_forward = test_data.select(
        predictor_columns
    )

    y_train_forward = train_data[
        target_column
    ].to_numpy()

    forward_imputer = SimpleImputer(
        strategy="median"
    )

    X_train_forward_imputed = forward_imputer.fit_transform(
        X_train_forward.to_numpy()
    )

    X_test_forward_imputed = forward_imputer.transform(
        X_test_forward.to_numpy()
    )

    forward_rf = RandomForestRegressor(
        n_estimators=500,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    forward_rf.fit(
        X_train_forward_imputed,
        y_train_forward
    )

    forward_yards = forward_rf.predict(
        X_test_forward_imputed
    )

    season_results = (
        test_data
        .select([
            "player_id",
            "season",
            "receiving_yards",
            "next_season_receiving_yards"
        ])
        .with_columns([
            pl.Series(
                "predicted_receiving_yards",
                forward_yards
            ),
            (
                pl.col("season") + 1
            ).alias("outcome_season")
        ])
    )

    forward_predictions.append(
        season_results
    )

forward_predictions_df = pl.concat(
    forward_predictions
)

print(
    "Forward prediction rows:",
    forward_predictions_df.height
)

print(
    "Predictor season range:",
    forward_predictions_df["season"].min(),
    "to",
    forward_predictions_df["season"].max()
)

print(
    "Outcome season range:",
    forward_predictions_df["outcome_season"].min(),
    "to",
    forward_predictions_df["outcome_season"].max()
)

Forward prediction rows: 1976
Predictor season range: 2013 to 2024
Outcome season range: 2014 to 2025


In [7]:
# Prepare contract events for outcome season matching

contract_match_candidates = (
    wr_contract_history
    .filter(
        pl.col("year_signed").is_not_null()
    )
    .select([
        "gsis_id",
        "player",
        "team",
        "contract_type",
        "status",
        "year_signed",
        "yrs",
        "total",
        "apy",
        "guarantees",
        "effective_apy"
    ])
)

prediction_contract_matches = (
    forward_predictions_df
    .join(
        contract_match_candidates,
        left_on=[
            "player_id",
            "outcome_season"
        ],
        right_on=[
            "gsis_id",
            "year_signed"
        ],
        how="left"
    )
)

matched_rows = (
    prediction_contract_matches
    .filter(
        pl.col("apy").is_not_null()
    )
    .height
)

matched_predictions = (
    prediction_contract_matches
    .filter(
        pl.col("apy").is_not_null()
    )
    .select([
        "player_id",
        "season",
        "outcome_season"
    ])
    .unique()
    .height
)

print(
    "Forward prediction rows:",
    forward_predictions_df.height
)

print(
    "Predictions with at least one same year contract match:",
    matched_predictions
)

print(
    "Total joined rows after contract matching:",
    prediction_contract_matches.height
)

print(
    "Rows containing contract data:",
    matched_rows
)

Forward prediction rows: 1976
Predictions with at least one same year contract match: 919
Total joined rows after contract matching: 2815
Rows containing contract data: 1758


In [8]:
# Measure contract ambiguity for matched predictions

contract_match_counts = (
    prediction_contract_matches
    .filter(
        pl.col("apy").is_not_null()
    )
    .group_by([
        "player_id",
        "season",
        "outcome_season"
    ])
    .agg(
        pl.len().alias("contract_matches"),
        pl.col("apy").n_unique().alias("unique_apy_values"),
        pl.col("team").n_unique().alias("unique_teams"),
        pl.col("contract_type").n_unique().alias("unique_contract_types")
    )
    .sort(
        "contract_matches",
        descending=True
    )
)

print(
    "Matched predictions:",
    contract_match_counts.height
)

print(
    "Matched predictions with more than one contract event:",
    contract_match_counts
    .filter(
        pl.col("contract_matches") > 1
    )
    .height
)

print(
    "Maximum contract events for one prediction:",
    contract_match_counts["contract_matches"].max()
)

contract_match_counts.head(20)

Matched predictions: 919
Matched predictions with more than one contract event: 366
Maximum contract events for one prediction: 8


player_id,season,outcome_season,contract_matches,unique_apy_values,unique_teams,unique_contract_types
str,i64,i64,u32,u32,u32,u32
"""00-0035639""",2023,2024,8,6,1,3
"""00-0032688""",2024,2025,8,3,2,2
"""00-0032128""",2021,2022,8,6,3,3
"""00-0036877""",2023,2024,8,5,2,2
"""00-0034867""",2021,2022,7,5,3,2
…,…,…,…,…,…,…
"""00-0031787""",2019,2020,6,4,3,3
"""00-0029000""",2021,2022,6,5,2,2
"""00-0035976""",2023,2024,6,4,2,2


In [9]:
# Examine contract types and statuses in ambiguous prediction matches

ambiguous_keys = (
    contract_match_counts
    .filter(pl.col("contract_matches") > 1)
    .select([
        "player_id",
        "season",
        "outcome_season"
    ])
)

ambiguous_contracts = (
    prediction_contract_matches
    .join(
        ambiguous_keys,
        on=[
            "player_id",
            "season",
            "outcome_season"
        ],
        how="inner"
    )
    .filter(
        pl.col("apy").is_not_null()
    )
)

ambiguous_type_status = (
    ambiguous_contracts
    .group_by([
        "contract_type",
        "status"
    ])
    .agg(
        pl.len().alias("events"),
        pl.col("player_id").n_unique().alias("unique_players"),
        pl.col("apy").median().alias("median_apy")
    )
    .sort(
        "events",
        descending=True
    )
)

print(
    "Ambiguous contract event rows:",
    ambiguous_contracts.height
)

print(
    "Contract type and status combinations:",
    ambiguous_type_status.height
)

ambiguous_type_status

Ambiguous contract event rows: 1205
Contract type and status combinations: 46


contract_type,status,events,unique_players,median_apy
str,str,u32,u32,f64
"""Practice""","""Elevated""",267,119,0.275334
"""SFA""","""Terminated""",225,144,0.91
"""Practice""","""Renegotiated""",154,116,0.257222
"""SFA""","""Expired""",135,111,0.99
"""Practice""","""Expired""",95,78,0.252
…,…,…,…,…
"""SFA""","""Reserve""",1,1,1.253285
"""ERFA""","""Claimed""",1,1,0.66
"""Extension""","""June 1 Terminated""",1,1,22.0


In [10]:
ambiguous_contracts.group_by("contract_type").agg(
    pl.len().alias("events"),
    pl.col("player_id").n_unique().alias("unique_players"),
    pl.col("apy").median().alias("median_apy"),
    pl.col("apy").max().alias("max_apy")
).sort("events", descending=True)

contract_type,events,unique_players,median_apy,max_apy
str,u32,u32,f64,f64
"""Practice""",592,174,0.2623335,0.684998
"""SFA""",385,194,0.96,15.25
"""UFA""",109,78,1.5,20.0
"""Extension""",50,26,17.125,32.0
"""ERFA""",26,20,0.705,3.0925
"""Other""",18,13,2.3,16.708333
"""Franchise""",13,5,21.816,28.75
"""RFA""",12,9,2.079,3.384


### Contract Selection Rule

Some players have more than one contract event in the same outcome season, so one contract must be selected for each prediction.

Contract types are prioritized as:

1. Extension
2. Franchise
3. UFA
4. RFA
5. ERFA
6. Drafted
7. Other
8. SFA
9. Practice

If more than one contract remains at the same priority, the contract with the highest APY is selected.

In [11]:
# Select one economically meaningful contract per prediction

contract_priority = (
    pl.when(pl.col("contract_type") == "Extension").then(1)
    .when(pl.col("contract_type") == "Franchise").then(2)
    .when(pl.col("contract_type") == "UFA").then(3)
    .when(pl.col("contract_type") == "RFA").then(4)
    .when(pl.col("contract_type") == "ERFA").then(5)
    .when(pl.col("contract_type") == "Drafted").then(6)
    .when(pl.col("contract_type") == "Other").then(7)
    .when(pl.col("contract_type") == "SFA").then(8)
    .when(pl.col("contract_type") == "Practice").then(9)
    .otherwise(99)
    .alias("contract_priority")
)

ranked_contract_matches = (
    prediction_contract_matches
    .filter(pl.col("apy").is_not_null())
    .with_columns(contract_priority)
    .sort(
        [
            "player_id",
            "season",
            "outcome_season",
            "contract_priority",
            "apy"
        ],
        descending=[
            False,
            False,
            False,
            False,
            True
        ]
    )
)

selected_contracts = (
    ranked_contract_matches
    .unique(
        subset=[
            "player_id",
            "season",
            "outcome_season"
        ],
        keep="first",
        maintain_order=True
    )
)

print(
    "Matched predictions before selection:",
    matched_predictions
)

print(
    "Selected one contract per matched prediction:",
    selected_contracts.height
)

print(
    "Duplicate prediction keys after selection:",
    selected_contracts
    .group_by([
        "player_id",
        "season",
        "outcome_season"
    ])
    .agg(pl.len().alias("rows"))
    .filter(pl.col("rows") > 1)
    .height
)

selected_contracts.select([
    "player_id",
    "season",
    "outcome_season",
    "player",
    "team",
    "contract_type",
    "status",
    "apy",
    "guarantees",
    "predicted_receiving_yards"
]).head(10)

Matched predictions before selection: 919
Selected one contract per matched prediction: 919
Duplicate prediction keys after selection: 0


player_id,season,outcome_season,player,team,contract_type,status,apy,guarantees,predicted_receiving_yards
str,i64,i64,str,str,str,str,f64,f64,f64
"""00-0020337""",2013,2014,"""Steve Smith Sr.""","""Ravens""","""SFA""","""Expired""",3.5,3.5,660.616046
"""00-0020494""",2013,2014,"""Santana Moss""","""Redskins""","""UFA""","""Expired""",1.02,0.065,521.632414
"""00-0022044""",2014,2015,"""Andre Johnson""","""Colts""","""SFA""","""Terminated""",7.0,10.0,860.773647
"""00-0022044""",2015,2016,"""Andre Johnson""","""Titans""","""SFA""","""Terminated""",1.8675,0.0,450.421172
"""00-0022084""",2013,2014,"""Anquan Boldin""","""49ers""","""UFA""","""Expired""",6.0,5.5,881.34535
"""00-0022084""",2015,2016,"""Anquan Boldin""","""Lions""","""UFA""","""Expired""",2.75,0.75,693.068163
"""00-0022427""",2014,2015,"""Wes Welker""","""Rams""","""SFA""","""Expired""",1.17,0.0,348.358875
"""00-0022811""",2013,2014,"""Jerricho Cotchery""","""Panthers""","""UFA""","""Expired""",2.5,2.25,599.07061
"""00-0022921""",2014,2015,"""Larry Fitzgerald""","""Cardinals""","""Other""","""Extended""",11.0,22.0,729.702507


In [12]:
# Audit contract types not covered by the selection hierarchy

unranked_contract_types = (
    ranked_contract_matches
    .filter(
        pl.col("contract_priority") == 99
    )
    .group_by("contract_type")
    .agg(
        pl.len().alias("events")
    )
    .sort(
        "events",
        descending=True
    )
)

print(
    "Contract events assigned fallback priority 99:",
    unranked_contract_types["events"].sum()
    if unranked_contract_types.height > 0
    else 0
)

print(unranked_contract_types)

Contract events assigned fallback priority 99: 0
shape: (0, 2)
┌───────────────┬────────┐
│ contract_type ┆ events │
│ ---           ┆ ---    │
│ str           ┆ u32    │
╞═══════════════╪════════╡
└───────────────┴────────┘


In [13]:
# Audit selected contract and prediction values before scoring

selected_contract_audit = selected_contracts.select([
    pl.len().alias("rows"),
    pl.col("predicted_receiving_yards").is_null().sum().alias(
        "missing_predictions"
    ),
    pl.col("apy").is_null().sum().alias(
        "missing_apy"
    ),
    (pl.col("apy") <= 0).sum().alias(
        "nonpositive_apy"
    ),
    pl.col("predicted_receiving_yards").min().alias(
        "min_prediction"
    ),
    pl.col("predicted_receiving_yards").max().alias(
        "max_prediction"
    ),
    pl.col("apy").min().alias(
        "min_apy"
    ),
    pl.col("apy").max().alias(
        "max_apy"
    )
])

print(selected_contract_audit)

shape: (1, 8)
┌──────┬──────────────┬─────────────┬──────────────┬─────────────┬─────────────┬─────────┬─────────┐
│ rows ┆ missing_pred ┆ missing_apy ┆ nonpositive_ ┆ min_predict ┆ max_predict ┆ min_apy ┆ max_apy │
│ ---  ┆ ictions      ┆ ---         ┆ apy          ┆ ion         ┆ ion         ┆ ---     ┆ ---     │
│ u32  ┆ ---          ┆ u32         ┆ ---          ┆ ---         ┆ ---         ┆ f64     ┆ f64     │
│      ┆ u32          ┆             ┆ u32          ┆ f64         ┆ f64         ┆         ┆         │
╞══════╪══════════════╪═════════════╪══════════════╪═════════════╪═════════════╪═════════╪═════════╡
│ 919  ┆ 0            ┆ 0           ┆ 0            ┆ 24.126321   ┆ 1313.041404 ┆ 0.136   ┆ 40.25   │
└──────┴──────────────┴─────────────┴──────────────┴─────────────┴─────────────┴─────────┴─────────┘


In [14]:
# Create outcome season contract and production benchmarks

season_benchmarks = (
    selected_contracts
    .group_by("outcome_season")
    .agg([
        pl.len().alias("matched_players"),
        pl.col("apy").median().alias("season_median_apy"),
        pl.col("predicted_receiving_yards").median().alias(
            "season_median_predicted_yards"
        )
    ])
    .sort("outcome_season")
)

print(season_benchmarks)

shape: (12, 4)
┌────────────────┬─────────────────┬───────────────────┬───────────────────────────────┐
│ outcome_season ┆ matched_players ┆ season_median_apy ┆ season_median_predicted_yards │
│ ---            ┆ ---             ┆ ---               ┆ ---                           │
│ i64            ┆ u32             ┆ f64               ┆ f64                           │
╞════════════════╪═════════════════╪═══════════════════╪═══════════════════════════════╡
│ 2014           ┆ 42              ┆ 3.0               ┆ 541.475987                    │
│ 2015           ┆ 53              ┆ 2.75              ┆ 378.796245                    │
│ 2016           ┆ 53              ┆ 1.5               ┆ 325.83101                     │
│ 2017           ┆ 64              ┆ 1.7333335         ┆ 348.243858                    │
│ 2018           ┆ 72              ┆ 1.5721865         ┆ 326.490343                    │
│ …              ┆ …               ┆ …                 ┆ …                             │
│ 2021

In [15]:
# Create season-relative contract value measures

gm_value = (
    selected_contracts
    .join(
        season_benchmarks,
        on="outcome_season",
        how="left"
    )
    .with_columns([
        (
            pl.col("apy")
            / pl.col("season_median_apy")
        ).alias("cost_index"),

        (
            pl.col("predicted_receiving_yards")
            / pl.col("season_median_predicted_yards")
        ).alias("production_index")
    ])
    .with_columns([
        (
            pl.col("production_index")
            / pl.col("cost_index")
        ).alias("value_index")
    ])
)

print(
    gm_value.select([
        "cost_index",
        "production_index",
        "value_index"
    ]).describe()
)

gm_value.select([
    "player",
    "team",
    "outcome_season",
    "contract_type",
    "apy",
    "predicted_receiving_yards",
    "cost_index",
    "production_index",
    "value_index"
]).head(10)

shape: (9, 4)
┌────────────┬────────────┬──────────────────┬─────────────┐
│ statistic  ┆ cost_index ┆ production_index ┆ value_index │
│ ---        ┆ ---        ┆ ---              ┆ ---         │
│ str        ┆ f64        ┆ f64              ┆ f64         │
╞════════════╪════════════╪══════════════════╪═════════════╡
│ count      ┆ 919.0      ┆ 919.0            ┆ 919.0       │
│ null_count ┆ 0.0        ┆ 0.0              ┆ 0.0         │
│ mean       ┆ 2.877191   ┆ 1.262954         ┆ 1.032165    │
│ std        ┆ 4.423595   ┆ 0.882116         ┆ 1.094351    │
│ min        ┆ 0.085      ┆ 0.100032         ┆ 0.046735    │
│ 25%        ┆ 0.691796   ┆ 0.597703         ┆ 0.430666    │
│ 50%        ┆ 1.0        ┆ 1.0              ┆ 0.75031     │
│ 75%        ┆ 2.775865   ┆ 1.717977         ┆ 1.221156    │
│ max        ┆ 29.700272  ┆ 4.840609         ┆ 11.891993   │
└────────────┴────────────┴──────────────────┴─────────────┘


player,team,outcome_season,contract_type,apy,predicted_receiving_yards,cost_index,production_index,value_index
str,str,i64,str,f64,f64,f64,f64,f64
"""Steve Smith Sr.""","""Ravens""",2014,"""SFA""",3.5,660.616046,1.166667,1.220028,1.045739
"""Santana Moss""","""Redskins""",2014,"""UFA""",1.02,521.632414,0.34,0.963353,2.833391
"""Andre Johnson""","""Colts""",2015,"""SFA""",7.0,860.773647,2.545455,2.272392,0.892726
"""Andre Johnson""","""Titans""",2016,"""SFA""",1.8675,450.421172,1.245,1.382377,1.110343
"""Anquan Boldin""","""49ers""",2014,"""UFA""",6.0,881.34535,2.0,1.627672,0.813836
"""Anquan Boldin""","""Lions""",2016,"""UFA""",2.75,693.068163,1.833333,2.127079,1.160225
"""Wes Welker""","""Rams""",2015,"""SFA""",1.17,348.358875,0.425455,0.919647,2.161564
"""Jerricho Cotchery""","""Panthers""",2014,"""UFA""",2.5,599.07061,0.833333,1.106366,1.327639
"""Larry Fitzgerald""","""Cardinals""",2015,"""Other""",11.0,729.702507,4.0,1.926372,0.481593


In [16]:
# Audit extreme contract value cases before defining GM categories

highest_value_cases = (
    gm_value
    .sort("value_index", descending=True)
    .select([
        "player",
        "team",
        "outcome_season",
        "contract_type",
        "status",
        "apy",
        "predicted_receiving_yards",
        "cost_index",
        "production_index",
        "value_index"
    ])
    .head(15)
)

lowest_value_cases = (
    gm_value
    .sort("value_index")
    .select([
        "player",
        "team",
        "outcome_season",
        "contract_type",
        "status",
        "apy",
        "predicted_receiving_yards",
        "cost_index",
        "production_index",
        "value_index"
    ])
    .head(15)
)

print("Highest value index cases:")
print(highest_value_cases)

print("\nLowest value index cases:")
print(lowest_value_cases)

Highest value index cases:
shape: (15, 10)
┌───────────┬──────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ player    ┆ team     ┆ outcome_s ┆ contract_ ┆ … ┆ predicted ┆ cost_inde ┆ productio ┆ value_ind │
│ ---       ┆ ---      ┆ eason     ┆ type      ┆   ┆ _receivin ┆ x         ┆ n_index   ┆ ex        │
│ str       ┆ str      ┆ ---       ┆ ---       ┆   ┆ g_yards   ┆ ---       ┆ ---       ┆ ---       │
│           ┆          ┆ i64       ┆ str       ┆   ┆ ---       ┆ f64       ┆ f64       ┆ f64       │
│           ┆          ┆           ┆           ┆   ┆ f64       ┆           ┆           ┆           │
╞═══════════╪══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Ray-Ray   ┆ Giants   ┆ 2025      ┆ Practice  ┆ … ┆ 517.33998 ┆ 0.18      ┆ 2.140559  ┆ 11.891993 │
│ McCloud   ┆          ┆           ┆           ┆   ┆ 6         ┆           ┆           ┆           │
│ Keke      ┆ Colts    ┆ 2021      ┆ Practice  ┆

In [17]:
# Create season relative percentile ranks for GM decision support

gm_value_ranked = (
    gm_value
    .with_columns([
        (
            pl.col("apy")
            .rank(method="average")
            .over("outcome_season")
            / pl.len().over("outcome_season")
        ).alias("cost_percentile"),

        (
            pl.col("predicted_receiving_yards")
            .rank(method="average")
            .over("outcome_season")
            / pl.len().over("outcome_season")
        ).alias("production_percentile")
    ])
    .with_columns([
        (
            pl.col("production_percentile")
            - pl.col("cost_percentile")
        ).alias("value_gap")
    ])
)

print(
    gm_value_ranked.select([
        "cost_percentile",
        "production_percentile",
        "value_gap"
    ]).describe()
)

gm_value_ranked.select([
    "player",
    "team",
    "outcome_season",
    "contract_type",
    "apy",
    "predicted_receiving_yards",
    "cost_percentile",
    "production_percentile",
    "value_gap"
]).sort(
    "value_gap",
    descending=True
).head(15)

shape: (9, 4)
┌────────────┬─────────────────┬───────────────────────┬────────────┐
│ statistic  ┆ cost_percentile ┆ production_percentile ┆ value_gap  │
│ ---        ┆ ---             ┆ ---                   ┆ ---        │
│ str        ┆ f64             ┆ f64                   ┆ f64        │
╞════════════╪═════════════════╪═══════════════════════╪════════════╡
│ count      ┆ 919.0           ┆ 919.0                 ┆ 919.0      │
│ null_count ┆ 0.0             ┆ 0.0                   ┆ 0.0        │
│ mean       ┆ 0.506529        ┆ 0.506529              ┆ 1.9329e-18 │
│ std        ┆ 0.288565        ┆ 0.288812              ┆ 0.212909   │
│ min        ┆ 0.010526        ┆ 0.008547              ┆ -0.757576  │
│ 25%        ┆ 0.261905        ┆ 0.257143              ┆ -0.111111  │
│ 50%        ┆ 0.5             ┆ 0.505263              ┆ 0.0        │
│ 75%        ┆ 0.757576        ┆ 0.757576              ┆ 0.111111   │
│ max        ┆ 1.0             ┆ 1.0                   ┆ 0.810526   │
└─────

player,team,outcome_season,contract_type,apy,predicted_receiving_yards,cost_percentile,production_percentile,value_gap
str,str,i64,str,f64,f64,f64,f64,f64
"""Ray-Ray McCloud""","""Giants""",2025,"""Practice""",0.315,517.339986,0.010526,0.821053,0.810526
"""Bryan Edwards""","""Chiefs""",2022,"""Practice""",0.207,438.749321,0.021368,0.769231,0.747863
"""Anthony Miller""","""Steelers""",2021,"""Practice""",0.252,473.402519,0.057143,0.771429,0.714286
"""Kenny Stills""","""Bills""",2020,"""Practice""",0.204,424.605806,0.060606,0.742424,0.681818
"""Allen Lazard""","""Packers""",2020,"""ERFA""",0.675,441.086959,0.143939,0.787879,0.643939
…,…,…,…,…,…,…,…,…
"""Kelvin Benjamin""","""Chiefs""",2018,"""SFA""",0.79,573.453956,0.270833,0.833333,0.5625
"""Brandon Johnson""","""Steelers""",2024,"""Practice""",0.351666,314.276979,0.042105,0.589474,0.547368
"""Dontrelle Inman""","""Chargers""",2016,"""ERFA""",0.6,414.936953,0.160377,0.698113,0.537736


### GM Decision Support Framework

The decision support categories compare each player's contract cost and expected production with other matched wide receivers in the same outcome season.

The primary categories are:

- **High-Cost / Lower-Production Review**: contract cost is in the top 25 percent of the season while predicted production is below the season median. These cases represent the clearest potential financial risk and warrant GM review.
- **High-Cost / High-Production**: both contract cost and predicted production are in the top 25 percent. These are expensive contracts accompanied by high expected production.
- **Lower-Cost / High-Production Value**: contract cost is below the season median while predicted production is in the top 25 percent. These cases may represent favorable production relative to financial commitment.
- **Market-Aligned / Other**: all remaining cases.

These categories are screening tools rather than automatic personnel recommendations. They identify contracts that merit additional review alongside age, guarantees, contract structure, roster role, injury information, team needs, and other factors not represented in the receiving yard model.

Practice contracts and other low cost transactional agreements are retained but should be interpreted in the context of their contract type.

In [18]:
# Assign GM decision support categories

gm_decision_support = (
    gm_value_ranked
    .with_columns(
        pl.when(
            (pl.col("cost_percentile") >= 0.75)
            & (pl.col("production_percentile") < 0.50)
        )
        .then(pl.lit("High-Cost / Lower-Production Review"))

        .when(
            (pl.col("cost_percentile") >= 0.75)
            & (pl.col("production_percentile") >= 0.75)
        )
        .then(pl.lit("High-Cost / High-Production"))

        .when(
            (pl.col("cost_percentile") < 0.50)
            & (pl.col("production_percentile") >= 0.75)
        )
        .then(pl.lit("Lower-Cost / High-Production Value"))

        .otherwise(pl.lit("Market-Aligned / Other"))
        .alias("gm_category")
    )
)

category_summary = (
    gm_decision_support
    .group_by("gm_category")
    .agg([
        pl.len().alias("players"),
        pl.col("apy").median().alias("median_apy"),
        pl.col("predicted_receiving_yards").median().alias(
            "median_predicted_yards"
        ),
        pl.col("value_gap").median().alias("median_value_gap")
    ])
    .sort("players", descending=True)
)

print(category_summary)

shape: (4, 5)
┌───────────────────────────────┬─────────┬────────────┬────────────────────────┬──────────────────┐
│ gm_category                   ┆ players ┆ median_apy ┆ median_predicted_yards ┆ median_value_gap │
│ ---                           ┆ ---     ┆ ---        ┆ ---                    ┆ ---              │
│ str                           ┆ u32     ┆ f64        ┆ f64                    ┆ f64              │
╞═══════════════════════════════╪═════════╪════════════╪════════════════════════╪══════════════════╡
│ Market-Aligned / Other        ┆ 719     ┆ 1.1275     ┆ 238.077326             ┆ 0.0              │
│ High-Cost / High-Production   ┆ 177     ┆ 11.0       ┆ 774.65805              ┆ 0.00641          │
│ Lower-Cost / High-Production  ┆ 13      ┆ 0.85       ┆ 479.747026             ┆ 0.5625           │
│ V…                            ┆         ┆            ┆                        ┆                  │
│ High-Cost / Lower-Production  ┆ 10      ┆ 4.875      ┆ 190.372156          

In [19]:
# Inspect primary GM contract review cases

gm_review_cases = (
    gm_decision_support
    .filter(
        pl.col("gm_category")
        == "High-Cost / Lower-Production Review"
    )
    .select([
        "player",
        "team",
        "outcome_season",
        "contract_type",
        "status",
        "yrs",
        "apy",
        "guarantees",
        "predicted_receiving_yards",
        "cost_percentile",
        "production_percentile",
        "value_gap"
    ])
    .sort(
        "value_gap"
    )
)

print(
    "Primary GM review cases:",
    gm_review_cases.height
)

gm_review_cases

Primary GM review cases: 10


player,team,outcome_season,contract_type,status,yrs,apy,guarantees,predicted_receiving_yards,cost_percentile,production_percentile,value_gap
str,str,i64,str,str,i32,f64,f64,f64,f64,f64,f64
"""Matt Slater""","""Patriots""",2020,"""UFA""","""Expired""",2,2.65,2.375,47.954784,0.772727,0.015152,-0.757576
"""Allen Robinson""","""Bears""",2018,"""UFA""","""Expired""",3,14.0,18.0,208.112848,0.916667,0.25,-0.666667
"""Markus Wheaton""","""Bears""",2017,"""UFA""","""Terminated""",2,5.5,6.0,172.631464,0.8125,0.15625,-0.65625
"""Deonte Harty""","""Bills""",2023,"""UFA""","""Terminated""",2,4.75,4.75,107.248067,0.820513,0.179487,-0.641026
"""Andre Roberts""","""Texans""",2021,"""UFA""","""Terminated""",2,2.75,2.5,108.30049,0.752381,0.114286,-0.638095
"""Jamal Agnew""","""Jaguars""",2021,"""UFA""","""Expired""",3,4.75,4.0,146.898657,0.804762,0.238095,-0.566667
"""Tavon Austin""","""Cowboys""",2018,"""Other""","""Expired""",1,7.0,5.0,242.375812,0.8125,0.333333,-0.479167
"""Cedrick Wilson Jr.""","""Dolphins""",2023,"""Other""","""Expired""",1,5.0,5.0,221.610259,0.839744,0.448718,-0.391026
"""Marquise Brown""","""Chiefs""",2025,"""UFA""","""Expired""",1,7.0,6.5,208.796922,0.789474,0.442105,-0.347368


In [20]:
# Add current receiving context to primary GM review cases

review_context = (
    gm_decision_support
    .filter(
        pl.col("gm_category")
        == "High-Cost / Lower-Production Review"
    )
    .join(
        wr_model.select([
            "player_id",
            "season",
            "targets",
            "receptions",
            "receiving_yards",
            "receiving_tds",
            "games"
        ]),
        on=[
            "player_id",
            "season"
        ],
        how="left"
    )
    .select([
        "player",
        "team",
        "season",
        "outcome_season",
        "contract_type",
        "apy",
        "guarantees",
        "games",
        "targets",
        "receptions",
        "receiving_yards",
        "predicted_receiving_yards",
        "cost_percentile",
        "production_percentile",
        "value_gap"
    ])
    .sort("value_gap")
)

review_context

player,team,season,outcome_season,contract_type,apy,guarantees,games,targets,receptions,receiving_yards,predicted_receiving_yards,cost_percentile,production_percentile,value_gap
str,str,i64,i64,str,f64,f64,i64,i64,i64,i64,f64,f64,f64,f64
"""Matt Slater""","""Patriots""",2019,2020,"""UFA""",2.65,2.375,7,0,0,0,47.954784,0.772727,0.015152,-0.757576
"""Allen Robinson""","""Bears""",2017,2018,"""UFA""",14.0,18.0,1,1,1,17,208.112848,0.916667,0.25,-0.666667
"""Markus Wheaton""","""Bears""",2016,2017,"""UFA""",5.5,6.0,3,9,4,51,172.631464,0.8125,0.15625,-0.65625
"""Deonte Harty""","""Bills""",2022,2023,"""UFA""",4.75,4.75,4,2,2,13,107.248067,0.820513,0.179487,-0.641026
"""Andre Roberts""","""Texans""",2020,2021,"""UFA""",2.75,2.5,15,5,4,34,108.30049,0.752381,0.114286,-0.638095
"""Jamal Agnew""","""Jaguars""",2020,2021,"""UFA""",4.75,4.0,14,20,13,89,146.898657,0.804762,0.238095,-0.566667
"""Tavon Austin""","""Cowboys""",2017,2018,"""Other""",7.0,5.0,14,22,13,47,242.375812,0.8125,0.333333,-0.479167
"""Cedrick Wilson Jr.""","""Dolphins""",2022,2023,"""Other""",5.0,5.0,13,18,12,136,221.610259,0.839744,0.448718,-0.391026
"""Marquise Brown""","""Chiefs""",2024,2025,"""UFA""",7.0,6.5,2,15,9,91,208.796922,0.789474,0.442105,-0.347368


### Interpretation of GM Review Flags

The High Cost / Lower Production Review category identifies contracts where cost ranks much higher than predicted receiving production for the same season. It is a review flag, not a conclusion that a player is overpaid.

The flagged cases also show an important limitation. Receiving yards do not capture special teams, blocking, leadership, roster role, or other contributions. Limited games and targets can also affect the forecast.

The review list should be used with contract structure, player role, availability, age, injury information, and roster needs before making a personnel decision.

In [21]:
# final GM decision-support table

gm_decision_table = (
    gm_decision_support
    .join(
        wr_model.select([
            "player_id",
            "season",
            "games",
            "targets",
            "receptions",
            "receiving_yards",
            "receiving_tds"
        ]),
        on=[
            "player_id",
            "season"
        ],
        how="left"
    )
    .select([
        "player",
        "team",
        "season",
        "outcome_season",
        "contract_type",
        "status",
        "yrs",
        "apy",
        "guarantees",
        "games",
        "targets",
        "receptions",
        "receiving_yards",
        "predicted_receiving_yards",
        "cost_percentile",
        "production_percentile",
        "value_gap",
        "gm_category"
    ])
    .sort([
        "outcome_season",
        "value_gap"
    ])
)

print(
    "GM decision-support rows:",
    gm_decision_table.height
)

print(
    "Outcome seasons:",
    gm_decision_table["outcome_season"].min(),
    "to",
    gm_decision_table["outcome_season"].max()
)

print(
    "Primary review cases:",
    gm_decision_table
    .filter(
        pl.col("gm_category")
        == "High-Cost / Lower-Production Review"
    )
    .height
)

gm_decision_table.tail(15)

GM decision-support rows: 919
Outcome seasons: 2014 to 2025
Primary review cases: 10


player,team,season,outcome_season,contract_type,status,yrs,apy,guarantees,games,targets,receptions,receiving_yards,predicted_receiving_yards,cost_percentile,production_percentile,value_gap,gm_category
str,str,i64,i64,str,str,i32,f64,f64,i64,i64,i64,i64,f64,f64,f64,f64,str
"""Olamide Zaccheaus""","""Bears""",2024,2025,"""UFA""","""Expired""",1,1.5,0.75,16,64,45,506,383.742085,0.473684,0.673684,0.2,"""Market-Aligned / Other"""
"""Ryan Flournoy""","""Cowboys""",2024,2025,"""SFA""","""Active""",2,1.0175,0.0,9,14,10,102,145.356926,0.105263,0.305263,0.2,"""Market-Aligned / Other"""
"""Bo Melton""","""Packers""",2024,2025,"""ERFA""","""Expired""",1,1.03,0.0,13,17,8,91,161.540122,0.152632,0.357895,0.205263,"""Market-Aligned / Other"""
"""David Moore""","""Panthers""",2024,2025,"""UFA""","""Expired""",1,1.4225,0.5635,15,57,32,351,357.698111,0.442105,0.652632,0.210526,"""Market-Aligned / Other"""
"""Allen Lazard""","""Jets""",2024,2025,"""Other""","""Terminated""",1,2.5,1.75,12,60,37,530,501.454231,0.594737,0.810526,0.215789,"""Market-Aligned / Other"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Jalen Brooks""","""Cardinals""",2024,2025,"""SFA""","""Active""",2,1.0875,0.0,12,30,12,177,247.877112,0.2,0.515789,0.315789,"""Market-Aligned / Other"""
"""Tay Martin""","""Commanders""",2024,2025,"""SFA""","""Terminated""",1,0.84,0.0,1,2,1,49,173.179858,0.052632,0.378947,0.326316,"""Market-Aligned / Other"""
"""Kevin Austin Jr.""","""Saints""",2024,2025,"""SFA""","""Active""",2,1.0175,0.0,8,22,11,151,215.817539,0.105263,0.452632,0.347368,"""Market-Aligned / Other"""


In [22]:
# Save final GM decision-support dataset

output_path = (
    processed_dir
    / "wr_gm_decision_support.csv"
)

gm_decision_table.write_csv(
    output_path
)

print(
    "Saved GM decision-support dataset:"
)

print(output_path)

print(
    "Saved rows:",
    gm_decision_table.height
)

Saved GM decision-support dataset:
..\data\processed\wr_gm_decision_support.csv
Saved rows: 919


In [23]:
# Validate saved GM decision-support dataset

gm_decision_check = pl.read_csv(
    processed_dir / "wr_gm_decision_support.csv"
)

print(
    "Reloaded shape:",
    gm_decision_check.shape
)

print(
    "Reloaded outcome season range:",
    gm_decision_check["outcome_season"].min(),
    "to",
    gm_decision_check["outcome_season"].max()
)

print(
    "Reloaded primary review cases:",
    gm_decision_check
    .filter(
        pl.col("gm_category")
        == "High-Cost / Lower-Production Review"
    )
    .height
)

Reloaded shape: (919, 18)
Reloaded outcome season range: 2014 to 2025
Reloaded primary review cases: 10


In [24]:
# Final checks

final_audit = pl.read_csv(
    processed_dir / "wr_gm_decision_support.csv"
)

duplicate_keys = (
    final_audit
    .group_by([
        "player",
        "season",
        "outcome_season"
    ])
    .agg(
        pl.len().alias("rows")
    )
    .filter(
        pl.col("rows") > 1
    )
    .height
)

print("Final rows:", final_audit.height)
print("Final columns:", final_audit.width)
print("Duplicate player-season-outcome keys:", duplicate_keys)
print(
    "Missing GM categories:",
    final_audit["gm_category"].null_count()
)

print("\nGM category counts:")

print(
    final_audit
    .group_by("gm_category")
    .agg(
        pl.len().alias("rows")
    )
    .sort(
        "rows",
        descending=True
    )
)

Final rows: 919
Final columns: 18
Duplicate player-season-outcome keys: 0
Missing GM categories: 0

GM category counts:
shape: (4, 2)
┌─────────────────────────────────┬──────┐
│ gm_category                     ┆ rows │
│ ---                             ┆ ---  │
│ str                             ┆ u32  │
╞═════════════════════════════════╪══════╡
│ Market-Aligned / Other          ┆ 719  │
│ High-Cost / High-Production     ┆ 177  │
│ Lower-Cost / High-Production V… ┆ 13   │
│ High-Cost / Lower-Production R… ┆ 10   │
└─────────────────────────────────┴──────┘


In [25]:
# Audit GM category assignment

category_audit = (
    gm_decision_support
    .select([
        pl.len().alias("total_rows"),
        pl.col("gm_category").is_null().sum().alias("missing_categories"),
        pl.col("gm_category").n_unique().alias("unique_categories"),
        pl.col("cost_percentile").min().alias("min_cost_percentile"),
        pl.col("cost_percentile").max().alias("max_cost_percentile"),
        pl.col("production_percentile").min().alias("min_production_percentile"),
        pl.col("production_percentile").max().alias("max_production_percentile")
    ])
)

print(category_audit)

print(
    gm_decision_support
    .group_by("gm_category")
    .agg(pl.len().alias("rows"))
    .sort("rows", descending=True)
)

shape: (1, 7)
┌────────────┬──────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ total_rows ┆ missing_cate ┆ unique_categ ┆ min_cost_pe ┆ max_cost_pe ┆ min_product ┆ max_product │
│ ---        ┆ gories       ┆ ories        ┆ rcentile    ┆ rcentile    ┆ ion_percent ┆ ion_percent │
│ u32        ┆ ---          ┆ ---          ┆ ---         ┆ ---         ┆ ile         ┆ ile         │
│            ┆ u32          ┆ u32          ┆ f64         ┆ f64         ┆ ---         ┆ ---         │
│            ┆              ┆              ┆             ┆             ┆ f64         ┆ f64         │
╞════════════╪══════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 919        ┆ 0            ┆ 4            ┆ 0.010526    ┆ 1.0         ┆ 0.008547    ┆ 1.0         │
└────────────┴──────────────┴──────────────┴─────────────┴─────────────┴─────────────┴─────────────┘
shape: (4, 2)
┌─────────────────────────────────┬──────┐
│ gm_category       

## Notebook 08 Summary

This notebook combined forward receiving yard predictions with wide receiver contract data to create a GM decision support framework. Contract variables were excluded from the performance model and introduced only after predictions were generated.

The analysis produced 1,976 forward predictions covering outcome seasons 2014 through 2025. Contract history contained 7,122 contract events across 1,626 wide receivers. Of the forward predictions, 919 matched at least one contract signed for the corresponding outcome season.

When multiple contracts matched a player and season, contracts were prioritized as Extension, Franchise, UFA, RFA, ERFA, Drafted, Other, SFA, and Practice. APY was used to break ties within the same priority. Contract status was excluded from selection to avoid using later outcomes in the historical decision process.

Contract cost and predicted production were ranked within each outcome season. The difference between production percentile and cost percentile was used to identify potential value or financial risk. The final framework classified 719 observations as Market Aligned / Other, 177 as High Cost / High Production, 13 as Lower Cost / High Production Value, and 10 as High Cost / Lower Production Review.

The review category is a screening tool, not a conclusion that a player is overpaid. Receiving yards do not capture special teams, blocking, leadership, availability, roster role, or other sources of player value.

The final dataset contains 919 observations and 18 fields and was saved as wr_gm_decision_support.csv.